# Introduction to Brightway — Part 1

In this section, we will discuss the fundamental concepts of Brightway. It is important to note that all this information is available online in the documentation page:

https://docs.brightway.dev/en/latest/index.html

## Install Brightway and the required dependencies

In [ ]:
!pip install bw2calc -q # Brightway package
!pip install bw2data -q # Brightway package
!pip install bw2io -q # Brightway package
!pip install polars -q
!pip install pypardiso -q
!pip install seaborn -q

<div class="alert alert-block alert-warning">
⚠️ You must restart the session! (Runtime > Restart Session)
</div>


## Configure your Brightway project
Because of the large size of the databases used in LCA, Brightway needs to write certain information to disk.
For this reason, every time a new project is created it must be configured.

The first step is to import the required dependencies:

In [ ]:
import bw2calc as bc
import bw2data as bd
import bw2io as bi
from rich import print

We can see the list of existing projects using the `bw2data` module:

In [ ]:
print("bw2data version: ", bd.__version__)
print("bw2io version: ", bi.__version__)
print("bw2calc version: ", bc.__version__)

In [ ]:
bd.projects

Any Python runtime environment that imports the `bw2data` package will be configured with the `default` project by default.


In [ ]:
bd.projects.current

If you want to switch projects, the function `bw2data.set_current(<your-project-name>)` allows you to select an existing project. If the project does not exist, this function will create a new project.

In [ ]:
bd.projects.set_current("nuevo_proyecto")

In [ ]:
# You can see that 'nuevo_proyecto' now appears in the project list.
bd.projects

<div class="alert alert-block alert-warning">
⚠️ All modifications made by the different Brightway modules are carried out EXCLUSIVELY within the context of the project. Therefore, it is important to verify that you are working with the correct project.
</div>


To keep a record of projects and other required information, `bw2data` writes some files to disk. There may be very exceptional cases where you need to access these files directly. You can locate them using the function `bw2data.projects.dir`.

If you want to make a copy of the current project, you can use `bw2data.projects.copy_project`.

In [ ]:
bd.projects.copy_project(new_name="nuevo_proyecto_2")

In [ ]:
# Check
bd.projects

If you want to delete a project, you can use the function `bw2data.projects.delete_dir`.

In [ ]:
# The `delete_dir` argument is Boolean and indicates
# whether you also want to delete the folder containing the project data.
bd.projects.delete_project(name="nuevo_proyecto", delete_dir=True)

🚧 **Hands-on**:
- Create a new project called `peru25`
- Create a copy of `peru25` called `peru25-prueba`
- Activate the `peru25` project


In [ ]:
# Insert the code here

In [ ]:
bd.projects

In [ ]:
bd.projects.set_current("example_project")
bd.databases

## Creating a new biosphere
Brightway is strongly, although not strictly, linked to the models and schemes used by ecoinvent.
For this reason, the impact methods and environmental flows (biosphere) are those provided by ecoinvent through its EcoQuery service.
Although the methods are developed by independent research groups, ecoinvent centralizes and modifies them so that they are compatible and ready to connect with its database.
This time, we will start by creating a biosphere by assigning the database to a new variable `biosfera` as follows:

In [ ]:
biosfera = bd.Database("biosphere3")
biosfera.register()
biosfera

## Manipulating databases
A database contains nodes, either from the biosphere or the technosphere. In other software, biosphere nodes are often called elementary flows, and technosphere nodes are called activities. In Brightway, the general concept of a `node` is used for any element contained in a database. This can be an elementary flow or a technosphere activity.

In this sense, a new database can be created as follows:

In [ ]:
# Primero, se asigna una instancia de base de datos a una variable
# Esta informacion esta en la memoria de la computadora pero no grabado en el disco
mi_db = bd.Database("mi_base_de_datos")

# Segundo, se registra la base de datos para que sea grabada en el disco
mi_db.register()

We can verify that there are now two databases: `biosphere3` and `mi_base_de_datos`, which we created ourselves.

In [ ]:
bd.databases

In many situations, it may be necessary to copy a database. This can be done as follows:

In [ ]:
new_database = bd.Database("biosphere3").copy("new_biosphere")

To delete a database, simply imagine that `bd.databases` has the same properties as a Python dictionary and use `del`.


In [ ]:
if "new_biosphere" in bd.databases:
    del bd.databases["new_biosphere"]

## Manipulating activities
One of Brightway's most important functionalities is creating activities, or nodes in general.
An activity can be created using the `new_activity` function, which belongs to database objects. In this case, any number of arguments can be provided, but the arguments `code`, `name`, `unit`, and `location` must ALWAYS be included. These four arguments are mandatory because they are the minimum required to have unique activities.


In [ ]:
bd.projects

In [ ]:
bd.databases

In [ ]:
if (
    "mi_base_de_datos" in bd.databases
):  # es una buena practica para siempre comenzar en un lienzo en blanco
    del bd.databases["mi_base_de_datos"]

In [ ]:
db = bd.Database("mi_base_de_datos")
db.register()
activity_ejemplo = db.new_activity(
    code="codigo-unico", name="nombre-no-unico", unit="unidad", location="PE"
)
activity_ejemplo.save()  # Este paso es SIEMPRE necesario para grabar la informacion en el disco
print(list(db))

This activity is now registered on disk and can be accessed using its `code` identifier and the `get` function. It is important to clarify that `code` is unique only within the database.

In [ ]:
actividad = db.get("codigo-unico")
print(actividad)

More detailed information about this activity can be viewed with the `as_dict` function, which returns a Python dictionary.

In [ ]:
actividad.as_dict()

If desired, the activity can be deleted using the `delete` function.


In [ ]:
actividad.delete()

The variable `actividad` refers to the activity with code `codigo-unico`. If you want to modify this activity, you will need to modify the content of the variable and then save it to disk.

In [ ]:
actividad["location"] = "DE"
actividad.save()
print(actividad.as_dict())

Following the bicycle example, we can create all the nodes, both technosphere and biosphere nodes.

In [ ]:
data = {
    "code": "bici",
    "name": "produccion bici",
    "location": "PE",
    "unit": "piece",
    "type": "processwithreferenceproduct",
}

bike = db.new_activity(**data)
bike.save()

data = {
    "code": "CF",
    "name": "carbon fibre",
    "unit": "kilogram",
    "location": "CN",
    "type": "processwithreferenceproduct",
}

cf = db.new_activity(**data)
cf.save()

ng = db.new_activity(
    name="Nat Gas",
    code="ng",
    location="NO",
    unit="MJ",
    type="processwithreferenceproduct",
)

ng.save()

print(list(db))

Now that the technosphere has been created, we will create nodes in the biosphere.

In [ ]:

# Create a node in the biosphere
co2 = bd.Database(
    "biosphere3"
).new_activity(
    name="Carbon Dioxide",
    code="co2",
    categories=(
        "air",
    ),  # Es importante definir las categorias cuando se crean nuevos methods debido a que
    type="emission",  # Es importante definir el tipo de flujo (emision or recurso)
)

co2.save()

In [ ]:
# # En caso quiera borrar todos los nodos de `db`
# co2.delete()
# for i in db:
# i.delete()

We now have all the nodes, but they are disconnected.
Without a connected network, we cannot compute the LCA. To do this, we need to create the `connections/interactions` between all nodes. In Brightway, these are called `exchanges`, and they can be created as follows with the `new_exchange` function:

In [ ]:

bike.new_exchange(amount=10, type="technosphere", input=cf).save()

cf.new_exchange(
    amount=237,
    type="technosphere",
    input=ng,
).save()

cf.new_exchange(
    amount=26.6,
    type="biosphere",
    input=co2,
).save()

If a flow needs to be modified, the data can be changed and saved to disk in the same way as for activities.

In [ ]:
print(list(bike.exchanges()))

In [ ]:
exc = list(bike.exchanges())[0]
exc.update(
    amount=2.5
)  # El metodo update funciona igual que con los diccionarios de python
exc.save()

We can now create a new method with only one characterization factor:

In [ ]:
ipcc = bd.Method(("IPCC",))  # Si no existe, lo crea
ipcc.write(
    [
        (co2.key, {"amount": 1}),
    ]
)

The `bw2calc` package contains the tools to perform calculations, such as the LCA class:

In [ ]:
lca = bc.LCA({bike: 1}, method=("IPCC",))  # Instancia la clase
lca.lci()  # calcula el inventario de ciclo de vida
lca.lcia()  # Calcula los impactos
print("The impact is: ", lca.score)

## Alternative method: Creating databases using dictionaries

In the previous section, we created a database sequentially:
1. First, we created each activity with `new_activity()`
2. Then, we created the exchanges with `new_exchange()`
3. Finally, we saved each element with `save()`

However, **Brightway also allows complete databases to be created using dictionaries**.
This method is more compact and useful when:
- You already have all the data structured
- You want to create multiple activities at once
- You are importing data from another source

The basic structure is:
```
db.write({
    (database_name, code): {
        'name': 'activity name',
        'unit': 'unit',
        'exchanges': [
            {'input': (db_name, code), 'amount': value, 'type': 'technosphere'},
            {'input': (db_name, code), 'amount': value, 'type': 'biosphere'}
        ]
    }
})
```

### Example: Recreating the bicycle model with dictionaries
Let's recreate the same bicycle model, but this time using the dictionary method:

In [ ]:
# First we create a new database
db_dict = bd.Database("mi_base_datos_dict")
db_dict.register()

# Now we write all activities and their exchanges at once
db_dict.write(
    {
        ("mi_base_datos_dict", "bici_dict"): {
            "name": "produccion bici (diccionario)",
            "unit": "piece",
            "location": "PE",
            "type": "processwithreferenceproduct",
            "exchanges": [
                {
                    "input": ("mi_base_datos_dict", "CF_dict"),
                    "amount": 2.5,  # Usamos el valor actualizado
                    "type": "technosphere",
                }
            ],
        },
        ("mi_base_datos_dict", "CF_dict"): {
            "name": "carbon fibre (diccionario)",
            "unit": "kilogram",
            "location": "CN",
            "type": "processwithreferenceproduct",
            "exchanges": [
                {
                    "input": ("mi_base_datos_dict", "ng_dict"),
                    "amount": 237,
                    "type": "technosphere",
                },
                {
                    "input": (
                        "biosphere3",
                        "co2",
                    ),  # Referencia al CO2 creado anteriormente
                    "amount": 26.6,
                    "type": "biosphere",
                },
            ],
        },
        ("mi_base_datos_dict", "ng_dict"): {
            "name": "Nat Gas (diccionario)",
            "unit": "MJ",
            "location": "NO",
            "type": "processwithreferenceproduct",
            "exchanges": [],  # Sin inputs en este ejemplo simple
        },
    }
)

In [ ]:
# Check that it was created correctly
print("Created activities:")
for act in db_dict:
    print(f"  - {act['name']}")

In [ ]:
# We can perform an LCA with this new database
bike_dict = bd.Database("mi_base_datos_dict").get("bici_dict")
lca_dict = bc.LCA({bike_dict: 1}, method=("IPCC",))
lca_dict.lci()
lca_dict.lcia()
print(f"El impacto usando el method de diccionarios es: {lca_dict.score}")
print(f"Comparado con el method secuencial: {lca.score}")

🚧 **Hands-on**:
- It has been discovered that carbon fiber production emits 0.23 kg of dinitrogen monoxide to air ($N_{2}O$) for each kilogram of carbon fiber produced.
- The characterization factor for $N_{2}O$ is 276.9
- By how much has the impact increased?

In [ ]:
# Create a node in the biosphere
n2o = bd.Database("biosphere3").new_activity(
    name="Nitrous oxide",
    code="n2o",
    categories=("air",),
    type="emission",
)

n2o.save()

cf.new_exchange(
    amount=0.23,
    type="biosphere",
    input=n2o,
).save()

In [ ]:
ipcc = bd.Method(("IPCC",))
factors = ipcc.load()  # se cargan los factores de caracterizacion
factors.append(((n2o.key), {"amount": 276.9}))  # Se agrega una nueva sustancia
ipcc.write(factors)

In [ ]:
ipcc.load()

In [ ]:
lca_nuevo = bc.LCA({bike: 1}, method=("IPCC",))  # Instancia la clase
lca_nuevo.lci()  # calcula el inventario de ciclo de vida
lca_nuevo.lcia()  # Calcula los impactos
print("The impact is now: ", lca_nuevo.score)

print(f"The impact increased by: {(lca_nuevo.score - lca.score) * 100 / lca.score} %")

## Exporting databases and projects
In the previous section, we learned how to create databases automatically (`biosphere3`) and manually (`mi_base_de_datos`).
In conventional situations, we often need to share our inventory models, either during collaborative work or to report our work to reviewers, colleagues, or for transparency reasons.
For this, `bw2io` offers a series of tools that can be used to export models in different formats.
For popularity reasons, in this section we will focus on three tools:
- Exporting a database to Excel
- Exporting a database to CSV (DataFrame)
- Exporting a project as a compressed backup file.

### Exporting to Excel
Brightway uses a template to read and export databases in Excel format. It is convenient for distributing final versions of the inventory. It is not very good at storing nested information. It does not allow changes to be tracked because `*.xlsx` is not a text format.

In [ ]:
# First, verify that you are in the correct project
bd.projects.current

In [ ]:
# If this is not the correct project, you know what to do
bd.projects.set_current("example_project")

In [ ]:
bi.export.excel.write_lci_excel??

In [ ]:
# dirpath is the argument que controla that controls where the file will be exported.
# En sistemas operativos tipo UNIX (Linux, MacOS), '.' means 'here'.
directorio = bi.export.excel.write_lci_excel(
    database_name="mi_base_de_datos", dirpath="."
)

### Exporting to CSV
Brightway allows nodes (activities) and edges (exchanges) to be converted into [pandas](https://pandas.pydata.org/) DataFrames.
A DataFrame is a tabular data structure that is widely used in data analysis and data science, and it can be exported directly as a CSV file.


In [ ]:
db.nodes_to_dataframe()  # Solo los nodos

In [ ]:
db.edges_to_dataframe()  # Solo aristas

In [ ]:
# The `to_csv` function es propia de pandas, not from Brightway
db.nodes_to_dataframe().to_csv("mis-nodos.csv")
db.edges_to_dataframe().to_csv("mis-aristas.csv")

### Exporting the full project as a backup
The last common option is to export the full project as a compressed archive. This is usually done when you want to save copies of all databases in a project. The disadvantage is that the resulting file can be large and is not suitable if you do not have permission to share commercial databases.

In [ ]:
bi.backup_project_directory("example_project", dir_backup=".")

## Importing private databases
This section is a natural continuation of the previous one, because we will simply learn how to import files that were previously exported. We will again assume that Excel, CSV, and `backup.tar.gz` are the only formats we are interested in.

### Importing an Excel file

In [ ]:
importador = bi.ExcelImporter("lci-mi_base_de_datos.xlsx")
importador.apply_strategies()
importador.match_database(
    fields=("name", "code", "unit", "location")
)  # Conecta nodos del archivo excel
importador.match_database(
    "biosphere3", fields=("name", "categories")
)  # Conecta nodos con la base de datos biosphere3
importador.statistics()
importador.write_excel()

In [ ]:
importador.match_database(
    "biosphere3", fields=("name", "unit", "categories")
)  # Conecta nodos con la base de datos biosphere3
importador.statistics()

In [ ]:
importador.write_database()
bd.databases  # La base de datos se ha importado correctamente

### Reproducing the results
Now we can simulate a reproducibility exercise and calculate the impacts once again.

In [ ]:
db = bd.Database("mi_base_de_datos")
bicicleta = db.get(
    "bici"
)  # seleccionamos la actividad que tiene codigo 'bici', la definimos en la seccion anterior

In [ ]:
lca = bc.LCA({bicicleta: 1}, method=("IPCC",))  # Instancia la clase
lca.lci()  # calcula el inventario de ciclo de vida
lca.lcia()  # Calcula los impactos
print("The impact is: ", lca.score)  # Es el mismo 🎉

🚧 **Hands-on**:
- A colleague has found an error in your model. The amount of natural gas consumed by the carbon fiber is not 237, but 23.7.
- Download the Excel file `lci-mi_base_de_datos.xlsx` to your personal computer and modify the value manually.
- Import the modified Excel file and recalculate the LCA. How much has the final impact changed?

In [ ]:
# Your code here